In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 60 — Hallucination-Reduction Fine-Tune Experiment

## Goal
Fine-tune a model on **grounded, fact-checked outputs** and compare with
a RAG-only approach. Explore whether fine-tuning can reduce hallucination.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Hallucination** | Model generating plausible but false info |
| **Grounded outputs** | Answers strictly based on provided context |
| **Fine-tune vs RAG** | Two approaches to reduce hallucination |
| **Faithfulness scoring** | Automated check: is the answer supported by context? |

## Approaches Compared
```
A. Base model (no context)      → Prone to hallucination
B. RAG-style (context in prompt) → Reduces hallucination via context
C. Fine-tuned (grounded data)    → Learns to only use given facts
```

## Domain: Product Knowledge Base
A company knowledge base with product specs, policies, and FAQs.

In [ ]:
import os, warnings, json, re
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Build a Knowledge Base

These are the **ground-truth facts** the model should use.
Any answer not supported by these facts is a hallucination.

In [ ]:
KNOWLEDGE_BASE = {
    "ProMax Laptop": {
        "price": "$1,299",
        "processor": "Intel Core i7-13700H",
        "ram": "16GB DDR5",
        "storage": "512GB NVMe SSD",
        "display": "15.6 inch IPS, 1920x1080, 144Hz",
        "battery": "72Wh, up to 8 hours",
        "weight": "1.8 kg",
        "warranty": "2 years standard, extendable to 3 years for $99",
    },
    "CloudSync Plan": {
        "free_tier": "5 GB storage, 1 device",
        "basic_plan": "$4.99/month, 100 GB, 3 devices",
        "pro_plan": "$9.99/month, 2 TB, unlimited devices",
        "encryption": "AES-256 at rest, TLS 1.3 in transit",
        "refund_policy": "30-day money-back guarantee, no questions asked",
    },
    "Return Policy": {
        "window": "30 days from delivery date",
        "condition": "Item must be in original packaging, unused",
        "exceptions": "Software and digital downloads are non-refundable",
        "process": "Submit return request online, receive prepaid shipping label within 24 hours",
        "refund_time": "5-7 business days after item received",
    },
}

def format_context(topic: str) -> str:
    """Format a KB entry as readable context."""
    if topic not in KNOWLEDGE_BASE:
        return "No information found."
    lines = [f"{topic}:"]
    for k, v in KNOWLEDGE_BASE[topic].items():
        lines.append(f"  - {k.replace('_', ' ').title()}: {v}")
    return "\n".join(lines)

for topic in KNOWLEDGE_BASE:
    print(format_context(topic))
    print()

## Step 2 — Build Faithfulness Eval

Score whether an answer is **faithful** to the context:
- Does every claim have support in the KB?
- Does the model refuse when info isn't available?

In [ ]:
eval_questions = [
    # Answerable from KB
    {"question": "What processor does the ProMax Laptop have?",
     "topic": "ProMax Laptop",
     "expected_facts": ["i7-13700H"],
     "should_answer": True},
    {"question": "How much does the CloudSync Pro plan cost?",
     "topic": "CloudSync Plan",
     "expected_facts": ["9.99"],
     "should_answer": True},
    {"question": "Can I return software purchases?",
     "topic": "Return Policy",
     "expected_facts": ["non-refundable"],
     "should_answer": True},
    {"question": "How long is the return window?",
     "topic": "Return Policy",
     "expected_facts": ["30 days"],
     "should_answer": True},
    {"question": "What encryption does CloudSync use?",
     "topic": "CloudSync Plan",
     "expected_facts": ["AES-256", "TLS"],
     "should_answer": True},
    {"question": "How heavy is the ProMax Laptop?",
     "topic": "ProMax Laptop",
     "expected_facts": ["1.8"],
     "should_answer": True},
    # Not in KB — model should say "I don't know" / refuse
    {"question": "What color options does the ProMax Laptop come in?",
     "topic": "ProMax Laptop",
     "expected_facts": [],
     "should_answer": False},
    {"question": "Does CloudSync support Linux?",
     "topic": "CloudSync Plan",
     "expected_facts": [],
     "should_answer": False},
    {"question": "Can I return items after 60 days?",
     "topic": "Return Policy",
     "expected_facts": [],
     "should_answer": False},
]

REFUSAL_PHRASES = [
    "i don't have", "not available", "no information",
    "i don't know", "not specified", "not mentioned",
    "cannot find", "unable to", "don't have that",
    "not provided", "beyond the", "outside the",
]

def score_faithfulness(response: str, expected_facts: list, should_answer: bool) -> dict:
    """
    Score a response for faithfulness.
    Returns dict with fact_recall, refusal_correct, and hallucination flag.
    """
    resp_lower = response.lower()
    is_refusal = any(phrase in resp_lower for phrase in REFUSAL_PHRASES)

    if should_answer:
        # Should contain expected facts
        found = sum(1 for f in expected_facts if f.lower() in resp_lower)
        fact_recall = found / len(expected_facts) if expected_facts else 1.0
        return {
            "fact_recall": fact_recall,
            "refusal_correct": None,
            "hallucinated": False,
        }
    else:
        # Should refuse or say "don't know"
        return {
            "fact_recall": None,
            "refusal_correct": is_refusal,
            "hallucinated": not is_refusal,
        }

def run_faithfulness_eval(predict_fn, eval_qs, model_name="model"):
    """Run full faithfulness evaluation."""
    results = []
    total_recall = []
    total_refusal = []

    for eq in eval_qs:
        response = predict_fn(eq["question"], eq["topic"])
        scores = score_faithfulness(response, eq["expected_facts"], eq["should_answer"])
        results.append({**eq, "response": response[:150], **scores})

        if scores["fact_recall"] is not None:
            total_recall.append(scores["fact_recall"])
        if scores["refusal_correct"] is not None:
            total_refusal.append(int(scores["refusal_correct"]))

    avg_recall = sum(total_recall) / len(total_recall) if total_recall else 0
    avg_refusal = sum(total_refusal) / len(total_refusal) if total_refusal else 0
    hallucination_count = sum(1 for r in results if r.get("hallucinated", False))

    print(f"\n{'='*55}")
    print(f" {model_name} — Faithfulness Results")
    print(f"{'='*55}")
    print(f" Fact Recall (answerable):    {avg_recall:.1%}")
    print(f" Refusal Rate (unanswerable): {avg_refusal:.1%}")
    print(f" Hallucinations:              {hallucination_count}/{len(total_refusal)}")
    print()
    for r in results:
        if r["should_answer"]:
            icon = "✅" if r["fact_recall"] == 1.0 else "⚠️"
            print(f"  {icon} Q: {r['question'][:50]}")
            print(f"     Recall: {r['fact_recall']:.0%} | A: {r['response'][:80]}")
        else:
            icon = "✅" if r["refusal_correct"] else "❌ HALLUCINATED"
            print(f"  {icon} Q: {r['question'][:50]}")
            print(f"     A: {r['response'][:80]}")

    return {"fact_recall": avg_recall, "refusal_accuracy": avg_refusal,
            "hallucinations": hallucination_count, "details": results}

print(f"Eval set: {len(eval_questions)} questions")
print(f"  Answerable: {sum(1 for q in eval_questions if q['should_answer'])}")
print(f"  Unanswerable (should refuse): {sum(1 for q in eval_questions if not q['should_answer'])}")

## Step 3 — Approach A: Base Model (No Context)

Ask the model questions **without** any supporting context.
This is the most hallucination-prone approach.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

def predict_no_context(question: str, topic: str) -> str:
    """Base model, no context — most prone to hallucination."""
    messages = [{"role": "user", "content": question}]
    t = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(t, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("Approach A: Base model with NO context")
results_no_ctx = run_faithfulness_eval(predict_no_context, eval_questions, "A. No Context")

## Step 4 — Approach B: RAG-Style (Context in Prompt)

Give the model relevant context **in the prompt**.
This is the RAG approach — no training needed.

In [ ]:
RAG_PROMPT = """Answer the question using ONLY the information below.
If the answer is not in the provided context, say "I don't have that information."
Do NOT make up facts.

Context:
{context}

Question: {question}

Answer:"""

def predict_rag(question: str, topic: str) -> str:
    """RAG-style: context included in prompt."""
    context = format_context(topic)
    prompt = RAG_PROMPT.format(context=context, question=question)
    messages = [{"role": "user", "content": prompt}]
    t = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(t, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("Approach B: RAG-style with context in prompt")
results_rag = run_faithfulness_eval(predict_rag, eval_questions, "B. RAG (Context in Prompt)")

## Step 5 — Approach C: Fine-Tune on Grounded Data

Train the model on examples where it:
1. Uses ONLY context facts when answering
2. Correctly refuses when answer isn't in context

In [ ]:
SYS_MSG = """You are a product support assistant. Answer questions using ONLY the provided context.
If the answer is not in the context, say "I don't have that information in our knowledge base."
Never make up or guess information."""

grounded_train = [
    # Answerable — correct grounded answers
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('ProMax Laptop')}\n\nQuestion: What is the battery life of the ProMax Laptop?"},
        {"role": "assistant", "content": "The ProMax Laptop has a 72Wh battery with up to 8 hours of battery life."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('ProMax Laptop')}\n\nQuestion: What display does the ProMax have?"},
        {"role": "assistant", "content": "The ProMax Laptop has a 15.6 inch IPS display with 1920x1080 resolution and a 144Hz refresh rate."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('ProMax Laptop')}\n\nQuestion: How much does the warranty extension cost?"},
        {"role": "assistant", "content": "The warranty can be extended from 2 years to 3 years for $99."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('CloudSync Plan')}\n\nQuestion: What's included in the free tier?"},
        {"role": "assistant", "content": "The CloudSync free tier includes 5 GB of storage and supports 1 device."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('CloudSync Plan')}\n\nQuestion: What is the refund policy for CloudSync?"},
        {"role": "assistant", "content": "CloudSync offers a 30-day money-back guarantee, no questions asked."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('Return Policy')}\n\nQuestion: How do I start a return?"},
        {"role": "assistant", "content": "Submit a return request online and you'll receive a prepaid shipping label within 24 hours."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('Return Policy')}\n\nQuestion: When will I get my refund?"},
        {"role": "assistant", "content": "Refunds are processed within 5-7 business days after the returned item is received."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('Return Policy')}\n\nQuestion: What condition must the item be in?"},
        {"role": "assistant", "content": "The item must be in its original packaging and unused."}]},
    # Unanswerable — correct refusals
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('ProMax Laptop')}\n\nQuestion: Does the ProMax Laptop come with a mouse?"},
        {"role": "assistant", "content": "I don't have that information in our knowledge base."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('ProMax Laptop')}\n\nQuestion: What GPU does it have?"},
        {"role": "assistant", "content": "I don't have that information in our knowledge base."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('CloudSync Plan')}\n\nQuestion: Can I share my account with family members?"},
        {"role": "assistant", "content": "I don't have that information in our knowledge base."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('CloudSync Plan')}\n\nQuestion: Is there a student discount?"},
        {"role": "assistant", "content": "I don't have that information in our knowledge base."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('Return Policy')}\n\nQuestion: Can I return a product bought from a third-party seller?"},
        {"role": "assistant", "content": "I don't have that information in our knowledge base."}]},
    {"messages": [{"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{format_context('Return Policy')}\n\nQuestion: Do you offer exchanges instead of refunds?"},
        {"role": "assistant", "content": "I don't have that information in our knowledge base."}]},
]

print(f"Grounded training examples: {len(grounded_train)}")
print(f"  Answerable: {sum(1 for t in grounded_train if 'I don\'t have' not in t['messages'][-1]['content'])}")
print(f"  Refusals:   {sum(1 for t in grounded_train if 'I don\'t have' in t['messages'][-1]['content'])}")

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Free base model memory
del base_model
torch.cuda.empty_cache()

OUTPUT_DIR = "./grounded_assistant"

trainer = SFTTrainer(
    model=MODEL_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=80,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        max_length=512,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=Dataset.from_list(grounded_train),
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print("Fine-tuning on grounded data...")
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"✅ Training done! Loss: {train_result.training_loss:.4f}")

## Step 6 — Eval the Fine-Tuned (Grounded) Model

In [ ]:
from peft import AutoPeftModelForCausalLM

ft_model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def predict_grounded(question: str, topic: str) -> str:
    """Fine-tuned model with context."""
    context = format_context(topic)
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    t = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(t, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("Approach C: Fine-tuned grounded model")
results_grounded = run_faithfulness_eval(predict_grounded, eval_questions, "C. Fine-Tuned (Grounded)")

## Step 7 — Three-Way Comparison

In [ ]:
import matplotlib.pyplot as plt

approaches = ["A. No Context", "B. RAG", "C. Fine-Tuned"]
recall_scores = [results_no_ctx["fact_recall"], results_rag["fact_recall"], results_grounded["fact_recall"]]
refusal_scores = [results_no_ctx["refusal_accuracy"], results_rag["refusal_accuracy"], results_grounded["refusal_accuracy"]]
halluc_counts = [results_no_ctx["hallucinations"], results_rag["hallucinations"], results_grounded["hallucinations"]]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["salmon", "skyblue", "mediumseagreen"]

# Fact recall
axes[0].bar(approaches, recall_scores, color=colors)
axes[0].set_title("Fact Recall (Answerable)")
axes[0].set_ylim(0, 1.15)
for i, v in enumerate(recall_scores):
    axes[0].text(i, v + 0.03, f"{v:.0%}", ha="center", fontweight="bold")

# Refusal accuracy
axes[1].bar(approaches, refusal_scores, color=colors)
axes[1].set_title("Refusal Accuracy (Unanswerable)")
axes[1].set_ylim(0, 1.15)
for i, v in enumerate(refusal_scores):
    axes[1].text(i, v + 0.03, f"{v:.0%}", ha="center", fontweight="bold")

# Hallucinations
axes[2].bar(approaches, halluc_counts, color=["crimson", "orange", "green"])
axes[2].set_title("Hallucinations")
axes[2].set_ylabel("Count")
for i, v in enumerate(halluc_counts):
    axes[2].text(i, v + 0.1, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Approach':<25} {'Fact Recall':<15} {'Refusal Acc':<15} {'Hallucinations':<15}")
print("─" * 70)
for i, name in enumerate(approaches):
    print(f"{name:<25} {recall_scores[i]:<15.1%} {refusal_scores[i]:<15.1%} {halluc_counts[i]}")

## 🧠 Key Concepts Recap

| Approach | Fact Recall | Hallucination Risk | Training Needed | Inference Cost |
|----------|------------|-------------------|-----------------|----------------|
| **No context** | Low | Very High | None | Low |
| **RAG (prompt)** | High | Medium | None | Medium (longer prompt) |
| **Fine-tuned** | High | Low | Yes | Low |

## When to Use Each
| Situation | Best Approach |
|-----------|---------------|
| KB changes frequently | RAG — no retraining |
| KB is stable, need low latency | Fine-tune |
| Budget for neither | Prompt engineering + RAG |
| Maximum safety | Fine-tune + RAG together |

## How Fine-Tuning Reduces Hallucination
```
Training on "I don't know" responses → Model learns to refuse
Training on context-grounded answers → Model learns to cite sources
Both together → Model prefers accuracy over helpfulness when unsure
```

## The Ideal Production Stack
```
User Query → Retrieve Context (RAG) → Fine-Tuned Model → Faithfulness Check → Response
                                         ↑                     ↑
                              Trained to ground          Post-hoc verification
                              & refuse gracefully
```